In [ ]:
%pip install kagglehub

import kagglehub

# Download latest version
path = kagglehub.dataset_download("arianazmoudeh/airbnbopendata")

print("Path to dataset files:", path)

In [ ]:
path


In [ ]:
import os
print(os.listdir(path))


In [ ]:
import pandas as pd

df = pd.read_csv(os.path.join(path, "Airbnb_Open_Data.csv"))
df.head()


In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 1. Drop unused columns

**Dropped:** `license` (almost all values missing), plus `house_rules`, `id`, `reviews per month`, `review rate number`, `calculated host listings count`, and `availability 365`.

**Why:** These columns are mostly empty or not needed for my analysis. I kept the other 18 columns.

In [ ]:
df.drop(columns=['license'], axis=1, inplace=True)

In [ ]:
df['house_rules'].head(100)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
columns_to_drop = ['reviews per month',
       'review rate number', 'calculated host listings count',
       'availability 365', 'house_rules','id']

In [ ]:
columns_to_keep = [c for c in df.columns if c not in columns_to_drop]


In [ ]:
columns_to_keep

In [ ]:
df

In [ ]:
df.columns


## 2. Standardize column names

Column names were mixed lowercase. I converted them all to uppercase so they are consistent.

In [ ]:
new_columns =[]
for i in df.columns:
    new_columns.append(i.upper())
    

In [ ]:
new_columns

In [ ]:
df


In [ ]:
df.rename(columns={df.columns[i]: new_columns[i] for i in range(len(new_columns))},inplace =True)



In [ ]:
df

## 3. Remove duplicates

**Problem:** 541 rows were exact copies of another row.

**Decision:** Dropped them with `drop_duplicates()`. Duplicates would inflate counts and averages.

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.shape


In [ ]:
df

## 4. Drop LAST REVIEW and rows without a name

**LAST REVIEW:** about 15,800 missing values, too empty to fill, so I dropped the column.

**NAME and HOST NAME:** I dropped rows where either was missing (about 650 rows, 0.6% of the data). These are identity text fields I cannot fill honestly.


In [ ]:
df = df.drop(columns = ['LAST REVIEW'], errors = 'ignore')

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
df = df.dropna(subset = ['NAME','HOST NAME'])

## 5. Missing HOST_IDENTITY_VERIFIED

**Problem:** 276 missing values.

**Check:** The split is 50.07% unconfirmed and 49.93% verified, almost exactly 50/50.

**Decision:** Fill with `unknown`. Mode would be a coin flip and would invent a status nobody gave.


In [ ]:
df['HOST_IDENTITY_VERIFIED']

In [ ]:
print(df["HOST_IDENTITY_VERIFIED"].isnull().sum())
print(df["HOST_IDENTITY_VERIFIED"].value_counts(normalize=True))

In [ ]:
df["HOST_IDENTITY_VERIFIED"] = df["HOST_IDENTITY_VERIFIED"].fillna('unknown')

In [ ]:
print(df["HOST_IDENTITY_VERIFIED"].isnull().sum())   # should be 0
print(df["HOST_IDENTITY_VERIFIED"].value_counts())

In [ ]:
df =df.copy()
print(df["HOST_IDENTITY_VERIFIED"].isnull().sum())

In [ ]:
df.isnull().sum()

## 6. Fix neighbourhood typos and fill missing locations

**Problem:** `NEIGHBOURHOOD GROUP` had typos (`brookln`, `manhatan`) and 26 missing values. `NEIGHBOURHOOD` had 16 missing.

**Decision:**
- Replaced the typos with the correct names.
- Filled missing groups from the neighbourhood, using a lookup built from rows that have both values (for example Harlem maps to Manhattan).
- Filled missing neighbourhoods with `unknown`, because a group does not tell me the exact neighbourhood.

In [ ]:
df["NEIGHBOURHOOD"].value_counts()

In [ ]:
df["NEIGHBOURHOOD GROUP"].value_counts()

In [ ]:
both = df['NEIGHBOURHOOD'].isnull() & df['NEIGHBOURHOOD GROUP'].isnull()
print(both.sum())

In [ ]:
df['NEIGHBOURHOOD GROUP'] = df['NEIGHBOURHOOD GROUP'].replace({
    "brookln":"Brooklyn",
    "manhatan": "Manhattan"
})

In [ ]:
print(df["NEIGHBOURHOOD GROUP"].value_counts())

In [ ]:
print(df["NEIGHBOURHOOD GROUP"].isnull().sum())
print(df["NEIGHBOURHOOD"].isnull().sum())

In [ ]:
known = df.dropna(subset=['NEIGHBOURHOOD','NEIGHBOURHOOD GROUP'])
lookup=known.groupby("NEIGHBOURHOOD")["NEIGHBOURHOOD GROUP"].agg(lambda x: x.mode()[0])
mask = df['NEIGHBOURHOOD GROUP'].isnull() & df['NEIGHBOURHOOD'].notnull()
df.loc[mask,'NEIGHBOURHOOD GROUP'] = df.loc[mask,'NEIGHBOURHOOD'].map(lookup)

In [ ]:
print(df["NEIGHBOURHOOD GROUP"].isnull().sum())
df["NEIGHBOURHOOD"] = df["NEIGHBOURHOOD"].fillna("unknown")
print(df["NEIGHBOURHOOD"].isnull().sum())   # should be 0

In [ ]:
df.isnull().sum() 

## 7. Missing COUNTRY and COUNTRY CODE

**Problem:** `COUNTRY` had 526 missing values and `COUNTRY CODE` had 122. 82 rows were missing both.


In [ ]:
print(df['COUNTRY'].value_counts())
print(df['COUNTRY CODE'].value_counts())

**Problem:** `COUNTRY` has 526 missing values and `COUNTRY CODE` has 122. 82 rows are missing both

In [ ]:
both2= df['COUNTRY'].isnull() & df['COUNTRY CODE'].isnull()
print(both2.sum())

## Checking 82 Neighbourhood

In [ ]:
no_country = df['COUNTRY'].isnull() & df['COUNTRY CODE'].isnull()
print(df[no_country]['NEIGHBOURHOOD GROUP'].value_counts())

 ## Fixing Missing COUNTRY and COUNTRY CODE

In [ ]:
df["COUNTRY"] = df["COUNTRY"].fillna("United States")
df["COUNTRY CODE"] = df["COUNTRY CODE"].fillna("US")
print(df["COUNTRY"].isnull().sum(), df["COUNTRY CODE"].isnull().sum())   # 0 0

In [ ]:
df.head(10)



In [ ]:
cols = ['INSTANT_BOOKABLE', 'CANCELLATION_POLICY']

for  c in cols:
    print(df[c].value_counts())

    

In [ ]:
df.isnull().sum()


## 8. Missing INSTANT_BOOKABLE and CANCELLATION_POLICY

**Problem:** 96 and 70 missing values.

**Check:** Both are almost evenly split (True and False about 50/50, three policies about 33% each).

**Decision:** Filled with `unknown`, for the same reason as `HOST_IDENTITY_VERIFIED`.

In [ ]:
print(df["INSTANT_BOOKABLE"].dtype)

In [ ]:
df["INSTANT_BOOKABLE"] = df["INSTANT_BOOKABLE"].fillna("unknown")
df["CANCELLATION_POLICY"] = df["CANCELLATION_POLICY"].fillna("unknown")

print(df["INSTANT_BOOKABLE"].value_counts())
print(df["CANCELLATION_POLICY"].value_counts())

## 9. PRICE and SERVICE FEE: text to numbers

**Problem:** Both were stored as text (`$966`), so I could not do math on them.

**Decision:** Removed the `$` and commas, then converted to numbers. Prices range from 50 to 1,200 and fees from 10 to 240, with no impossible values.

In [ ]:
print(df[["PRICE", "SERVICE FEE", "MINIMUM NIGHTS","CONSTRUCTION YEAR"]].dtypes)
print(df[["PRICE", "SERVICE FEE", "MINIMUM NIGHTS","CONSTRUCTION YEAR"]].head())

In [ ]:
columnss = ['SERVICE FEE',"PRICE"]
for c in columnss:
    df[c] =(df[c].astype(str)
                .str.replace(",","",regex=False)
                .str.replace("$","",regex=False)
                .str.strip()
                )
    df[c] =pd. to_numeric(df[c], errors='coerce')

In [ ]:
df.head(10)

In [ ]:
print(df[["PRICE", "SERVICE FEE"]].dtypes)
print(df[["PRICE", "SERVICE FEE"]].isnull().sum())


In [ ]:
print([df["PRICE"].min(), df["PRICE"].max()])
print([df["SERVICE FEE"].min(), df["SERVICE FEE"].max()])

In [ ]:
print(df[['PRICE','SERVICE FEE']].describe())



In [ ]:
print((df["SERVICE FEE"] / df["PRICE"]).describe())


## Handling missing PRICE and SERVICE FEE

**PRICE:** Dropped rows with missing price. Price is the likely target, so I don't want to invent values for it.

**SERVICE FEE:** Filled missing fees with `PRICE * 0.2`, rounded. This is based on the 20% ratio checked above (std 0.001).

**Check:** Both columns now have 0 missing values.

In [ ]:
df = df.dropna(subset=["PRICE"]).copy()
df["SERVICE FEE"] = df["SERVICE FEE"].fillna((df["PRICE"] * 0.2).round())

print(df["PRICE"].isnull().sum())    
print(df["PRICE"].dtypes)      # 0
print(df["SERVICE FEE"].isnull().sum())   # 0

### Safety check: decimal prices

Real prices here are whole dollars. This cell removes any row whose price has a decimal. It removed **0 rows**, which confirms no invented prices (like the old mean of 625.381) remain.

In [ ]:
before = len(df)
df = df[df["PRICE"] % 1 == 0].copy()
print(before, "->", len(df), "rows removed:", before - len(df)) 

In [ ]:
for c in ["PRICE", "SERVICE FEE"]:
    df[c] = df[c].astype("Int64")

print(df[["PRICE", "SERVICE FEE"]].dtypes)

In [ ]:
df.head(10)

## 11. CONSTRUCTION YEAR, MINIMUM NIGHTS, NUMBER OF REVIEWS

**Types:** These were `float64` (like 2015.0) because of missing values. I converted them to `Int64`.

**MINIMUM NIGHTS problem:** Impossible values: 13 negative and 33 above 365 (min -1223, max 5645).

**Decision:** Treated values below 1 or above 365 as errors, replaced them with missing, then filled all missing values with the median (3). The mean (8.1) is pulled up by extreme values.

**CONSTRUCTION YEAR and NUMBER OF REVIEWS:** Filled with the median. Years run from 2003 to 2022, and reviews are skewed, so median is safer.

**Assumption:** 365 is my own cutoff. Only 33 rows are above it.


In [ ]:
cols = ["CONSTRUCTION YEAR", "MINIMUM NIGHTS", "NUMBER OF REVIEWS"]

for c in cols:
    df[c] = df[c].astype("Int64")

print(df[cols].dtypes)
print(df[cols].head())

In [ ]:
cols = ["CONSTRUCTION YEAR", "MINIMUM NIGHTS", "NUMBER OF REVIEWS"]

print(df[cols].isnull().sum())
print(df[cols].describe())


In [ ]:
print((df["MINIMUM NIGHTS"] < 1).sum(),"below 1 ")
print((df["MINIMUM NIGHTS"] > 365).sum(),"above 365 ")
print(df["MINIMUM NIGHTS"].value_counts().sort_index().head(10))
print(df["MINIMUM NIGHTS"].value_counts().sort_index().tail(10))


In [ ]:
bad = (df["MINIMUM NIGHTS"] < 1) | (df["MINIMUM NIGHTS"] > 365)
print(bad.sum(), "rows to fix")          # should print 46

df.loc[bad, "MINIMUM NIGHTS"] = pd.NA
df["MINIMUM NIGHTS"] = df["MINIMUM NIGHTS"].fillna(int(df["MINIMUM NIGHTS"].median()))

print(df["MINIMUM NIGHTS"].describe())   # min >= 1, max <= 365
print(df["MINIMUM NIGHTS"].isnull().sum())   # should be 0

In [ ]:
df["CONSTRUCTION YEAR"] = df["CONSTRUCTION YEAR"].fillna(int(df["CONSTRUCTION YEAR"].median()))
df["NUMBER OF REVIEWS"] = df["NUMBER OF REVIEWS"].fillna(int(df["NUMBER OF REVIEWS"].median()))

print(df[["CONSTRUCTION YEAR", "MINIMUM NIGHTS", "NUMBER OF REVIEWS"]].isnull().sum())   # all 0

## 12. LATITUDE and LONGITUDE

Renamed `LAT` and `LONG` to full names. Dropped the 8 rows with missing coordinates, because filling with the mean would place listings at a made-up spot.

In [ ]:
df = df.rename(columns={"LAT": "LATITUDE", "LONG": "LONGITUDE"})


In [ ]:
df = df.dropna(subset=["LATITUDE"])
df = df.dropna(subset=["LONGITUDE"])
print(df[["LATITUDE", "LONGITUDE"]].isnull().sum())   # should be 0

In [ ]:
df.head(10)

In [ ]:
df.dtypes


## 13. Final check

No missing values, no duplicates, 5 neighbourhood groups, and clean room types. The final shape is 101,163 rows and 18 columns (about 1.4% of rows removed overall).



In [ ]:
print(df.isnull().sum())
print(df.shape)
print(df.duplicated().sum())
print(df["NEIGHBOURHOOD GROUP"].value_counts())
print(df["ROOM TYPE"].value_counts())

In [ ]:
df.head(5)

### Save to Excel File

In [ ]:
pip install openpyxl


In [ ]:
df.to_excel("cleaned_airbnb_data.xlsx", index=False)